# Calvin Site 1 • RLD search with a whole-vine mass constraint

Open from the project with the local research bundle available. Read saved results at both observation cutoffs and inspect the primary diameter evidence. Search ranks use measured core positions and depth increments. Each listed call regenerates a full-resolution vine.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
study = root / '.runtime/calvin-rld-search-20261119'
if not (study / 'FROZEN.json').is_file():
    raise FileNotFoundError(f'This review needs the local study bundle at {study}')
read = lambda path: json.loads(path.read_text())
display(Markdown((study / 'REPORT.md').read_text()))


## Standing-vine evidence and candidate priors

Compare the sampled basal envelope to the proximal mid-exposure diameters. The distance along each root to the field measurement is unknown.

In [ ]:
field = read(study / 'FIELD_EVIDENCE.json')
display({key:field[key] for key in ('count','angle_missing','mean_mm','sd_mm','min_mm','max_mm','top_two_area_fraction')})
display(pd.read_csv(study / 'primary_diameter_prior_checks.csv').groupby('stage').agg({'largest_possible_basal_mm':['min','max'],'chance_any_basal_at_least_47mm':['min','max']}))


## Essential parameters to optimize

Each per-order path uses `features.<feature>.settings.by_order.<order>`.

| Start with | Schema-4 settings | Read the response in |
|---|---|---|
| Primary diameter and taper | Order 1 `taper` → `base_cm`, `rate_per_cm`. The initial grid crosses basal distribution factors 1.00, 1.25, 1.50 and 1.75 with about 0.0035, 0.00657 and 0.00875 cm⁻¹. | Whole-vine mass, local daughter eligibility and RLD. |
| Length and branching | `lengths` distributions by order. In `branching`, adjust `daughter_probabilities` for parent orders 3–4 or `mean_spacing_cm` for parent orders 1–2 in the Poisson family. Consider separate order-2 and order-3 diameter changes later. | RLD magnitude and its distribution over depth and between cores. |
| Direction | In `curvature`, vary `row_alignment_per_cm` and `downward_drift_per_cm` on orders 1–3. Explore `downward_drift_per_cm` and `insertion_mean_deg` on orders 4–5. | Along-row spread, RLD depth profile and joint core-by-depth allocation. |

For the initial comparison, hold the nominal primary contour mean near 107.5 cm and its upper bound at 250 cm. Hold density at 0.45 g/cm³, the local parent gate at 0.093 cm and the daughter cap at 0.95. Compare measured and sampled crowns as named cases. Score RLD in the 0.93–2.50 mm band and use 0.50–2.50 mm for sensitivity. Check total dry mass against 5–7 kg for one vine, including its separately estimated underground crown/rootstock.

## Feasibility at the sampled core locations

An RLD ratio of 1 means the predicted mean matches the observed mean. Lower `cv_score` is better. `mass_in_band` gives the fraction of realizations within the mass band. Compare candidates with the same seed panels.

In [ ]:
ledger = pd.read_csv(study / 'all_attempts.csv')
display(ledger.groupby(['stage','status']).size().rename('attempts').reset_index())
ranking = pd.read_csv(study / 'development_frontier.csv')
display(ranking[['stage','case','seeds','cv_score','ratio','mass_mean_kg','mass_in_band','both_mean_guards']].head(20))
display(Image(filename=str(study / 'development_frontier.png'), width=720))


## Does a wider core change the spatial fit?

Compare the same simulated roots at the same centres and 10 cm depth increments. The physical core uses the soil volume in the workbook. The two virtual cylinders use 3.8 cm and 7.6 cm diameters. Each predicted length is divided by that cylinder’s own soil volume. The observed RLD and the number of observed cores stay fixed.

Read `ratio` for predicted/observed mean RLD, `joint` for differences across core-by-depth cells, and `depth` for differences across depth layers. `zero_fraction` counts model misses across individual realizations. Compare mean bias and loss as well as the change in misses. The optional recovery preference gives a small discount to slight overprediction. The actual fraction of roots lost in coring is unknown.

In [ ]:
area = study / 'area_support'
if (area / 'control_comparison.csv').is_file():
    area_controls = pd.read_csv(area / 'control_comparison.csv')
    display(area_controls[['case','support','seeds','ratio','joint','depth','zero_fraction',
                           'cv_score','cv_preference_score','mass_mean_kg']].round(4))
for file in ['C-training.csv','E-training.csv','confirmation.csv']:
    if (area / file).is_file():
        scores = pd.read_csv(area / file)
        if 'comparison_cores' in scores:
            scores = scores[scores.comparison_cores & scores.qc.eq('all') & scores.cutoff_mm.eq(0.93)]
        display(scores[['case','support','seeds','ratio','joint','depth','zero_fraction',
                        'cv_score','cv_preference_score','mass_mean_kg']].head(24))
if (area / 'control_profiles.png').is_file():
    display(Image(filename=str(area / 'control_profiles.png'), width=900))


## Neighbour overlap, spatial fit and tissue density

The left panel compares mean underground mass and physical-core RLD in the archived development panels. The right panel shows the QC reference at each development core on the wider support. Both predictions include the subject vine and eight translated neighbours. Each root length inside a cylinder contributes once. The total soil volume is that of one cylinder. The mass comparison includes only the subject vine and its underground crown.

The density plan compares 0.25, 0.35 and 0.45 g/cm³ on roots 1–2.5 mm with identical geometry. A separate treatment uses 0.30 for 1–2 mm and 0.40 for >2–2.5 mm. The 30% biomass target includes all subject roots ≤2.5 mm, including roots below 1 mm, with crown mass in the denominator. The historical search above uses 0.45 throughout.

[Density treatments, implementation and architecture search ranges](../../.runtime/calvin-site1-diagnosis-20261119/DENSITY_AND_FIT_PLAN.md) · [Open the local notebook with the saved figure](../../.runtime/calvin-site1-diagnosis-20261119/calvin_site1_rld_mass_search_with_diagnosis.ipynb)

In [ ]:
diagnosis = root / '.runtime' / 'calvin-site1-diagnosis-20261119'
figure = diagnosis / 'diagnosis.png'
if figure.is_file():
    display(Image(filename=str(figure), embed=True, width=980))
else:
    print(f'The local diagnostic figure is stored at {figure}.')


### What the archived mass records can establish

The saved trials give mass above and below 2.5 mm. The calculation below brackets the effect of changing only the 1–2.5 mm density. The lower bound assigns the entire fine compartment the test density. The upper bound keeps all fine material at 0.45. The table averages per-vine shares over the same seeds as the diagnostic figure. Exact values require recording the volume on each side of 1 mm. The separate stepped-density treatment also needs a split at 2 mm.

In [ ]:
from hashlib import sha256

receipt_index = diagnosis / 'source_receipts.csv'
density_cases = ['qc20', 'qc-d1.75-k0.00657210-disk', 'qc-d1.75-k0.00350000', 'C_003']
if receipt_index.is_file():
    index = pd.read_csv(receipt_index)
    bound_rows = []
    for candidate in density_cases:
        panel = index[index['case'].eq(candidate)]
        if panel.empty:
            raise ValueError(f'Missing diagnostic panel for {candidate}')
        for record in panel.itertuples(index=False):
            path = study / 'trials' / record.stage / candidate / f'{record.seed}.json'
            blob = path.read_bytes()
            assert sha256(blob).hexdigest() == record.trial_sha256
            run = json.loads(blob)
            summary = run['root_summary']
            assert run['status'] == 'passed' and summary['threshold_root_diameter_cm'] == 0.25
            fine = summary['diameter_classes']['at_or_below']['dry_biomass_g'] / 1000
            total = run['mass_total_kg']
            for density in (0.25, 0.35, 0.45):
                lightest_fine = fine * density / 0.45
                lightest_total = total - fine + lightest_fine
                bound_rows.append(dict(case=candidate, seed=record.seed, density=density,
                    total_low_kg=lightest_total, total_high_kg=total,
                    fine_low_pct=100 * lightest_fine / lightest_total, fine_high_pct=100 * fine / total))
    density_bounds = pd.DataFrame(bound_rows).groupby(['case', 'density'], sort=False).agg(
        seeds=('seed', 'nunique'), total_low_kg=('total_low_kg', 'mean'), total_high_kg=('total_high_kg', 'mean'),
        fine_low_pct=('fine_low_pct', 'mean'), fine_high_pct=('fine_high_pct', 'mean'))
    display(density_bounds.round(2))
else:
    print(f'The density bounds need the local receipt index at {receipt_index}.')


## Diameter-dependent tissue density and the wider spatial search

The local study compares intact model vines on matched seeds at the original core locations. All spatial predictions include the central vine and eight translated neighbours. The physical core and the 7.6 cm core use their respective soil volumes. The earlier depth-profile figure remains in the core-width section above.

The stepped law uses dry grams per cubic centimetre of fresh root tissue. Its diameter boundaries in mm are 1, 2, 2.5, 12 and 30. The six densities are 0.25, 0.30, 0.40, 0.45, 0.50 and 0.50. A 1 mm cylinder enters the 1–2 mm band. The other upper endpoints belong to the finer band. The whole-vine estimate includes a separately assumed crown mass. The local profile and spatial parameter choices are research assumptions.


In [ ]:
spatial_study = root / '.runtime' / 'calvin-density-spatial-20261119'
spatial_figure = spatial_study / 'spatial_fit.png'
if spatial_figure.is_file():
    display(Image(filename=str(spatial_figure), embed=True, width=1000))
if (spatial_study / 'performance.csv').is_file():
    scores = pd.read_csv(spatial_study / 'performance.csv')
    selected_scores = scores[scores.panel.eq('confirm') & scores.case.isin(['QC', 'control', 'selected'])]
    display(selected_scores[['case', 'split', 'n_cores', 'increments', 'phys_loss', 'wide_loss',
                             'phys_ratio', 'wide_ratio', 'phys_joint', 'wide_joint']].round(3))


### Applying the density law to a new simulation

Supply the band definition under `parameters.root_tissue_density` in a schema-4 configuration, then run the simulator normally. Each emitted segment carries the volume-weighted effective dry density. `sim.diameter_material()` returns exact per-order lengths, volumes and dry masses for the configured diameter bands. A scalar density is also supported. The local calibration recipe specifies the spatial terminal taper and crown-orientation averaging used in the search.


In [ ]:
stepped_density = dict(bounds_mm=[1., 2., 2.5, 12., 30.],
                       density_g_cm3=[.25, .30, .40, .45, .50, .50],
                       equality_to_finer=[False, True, True, True, True])
# For a new run: config["parameters"]["root_tissue_density"] = stepped_density
# sim.run(config)
# material_by_order = sim.diameter_material()["by_order"]
if (spatial_study / 'confirm_density_sensitivity.csv').is_file():
    sensitivity = pd.read_csv(spatial_study / 'confirm_density_sensitivity.csv')
    display(sensitivity[sensitivity.case.eq('selected')].groupby('scenario')[
        ['kg', 'fine_kg', 'share', 'root_only_share']].mean().round(3))
if (spatial_study / 'REPORT.md').is_file():
    display(Markdown((spatial_study / 'REPORT.md').read_text()))


In [ ]:
# Set a seed to replay the selected spatial prediction locally.
REPLAY_SPATIAL_SEED = None
if REPLAY_SPATIAL_SEED is not None:
    import subprocess
    python = root / '.runtime' / 'envs' / 'py310-qualification' / 'bin' / 'python'
    subprocess.run([str(python), str(spatial_study / 'replay_selected.py'),
                    '--seed', str(REPLAY_SPATIAL_SEED)], check=True, cwd=root)
    display(pd.read_csv(spatial_study / 'replay' / str(REPLAY_SPATIAL_SEED) / 'metrics.csv'))


## Confirmation and replay

The finalist list is frozen using development data before the fresh confirmation panel. The three comparison cores have been inspected before.

In [ ]:
if (study / 'RESULT.json').is_file():
    result = read(study / 'RESULT.json')
    display(result)
    display(pd.read_csv(study / 'uncertainty.csv'))
    display(Image(filename=str(study / 'held_rld_profiles.png'), width=900))
    cfg = read(study / 'inputs' / (result['selected_in_development'] + '.schema4.private.json'))
    display(cfg['parameters'])
else:
    print('Final confirmation evidence is pending. See the execution ledger above.')


## Rebuild the review from the receipts

The switch below recomputes tables without generating new roots. The local study includes the frozen configurations, code, receipts and private targets. From the repository root, run `bash .runtime/calvin-rld-search-20261119/run.sh supports.py` for the paired baseline replays and `bash .runtime/calvin-rld-search-20261119/run.sh area_search.py` for the area-aware search. Both acquire the same worker lock. Compare only complete matched seed panels.

In [ ]:
REBUILD_REVIEW = False
if REBUILD_REVIEW:
    import subprocess
    subprocess.run(['bash', str(study/'run.sh'), 'review.py'], cwd=root, check=True)
